In [1]:
import logging
import requests
import numpy as np
import faiss

from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph

from transformers import pipeline
from sentence_transformers import SentenceTransformer

c:\Users\vedan\Desktop\GenAIProject\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

In [3]:
class RepoState(TypedDict):

    owner: str
    repo: str

    files: List[str]
    filtered_files: List[str]

    documents: List[Dict]
    embeddings: List

    vector_index: object
    retrieved_chunks: List

    issues: List
    karma_score: float

In [6]:
logger.info("Loading TinyLlama model")

llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=-1,
    max_new_tokens=200,
    temperature=0.2,
    do_sample=False
)

2026-03-09 08:44:15,214 | INFO | Loading TinyLlama model
2026-03-09 08:44:16,001 | INFO | HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 08:44:16,111 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 910.31it/s, Materializing param=model.norm.weight]                              
2026-03-09 08:44:16,997 | INFO | HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 08:44:17,106 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
2026-03-09 08:44:17,485 | INFO | H

In [7]:
logger.info("Loading embedding model")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

2026-03-09 08:44:43,012 | INFO | Loading embedding model
2026-03-09 08:44:43,025 | INFO | Use pytorch device_name: cpu
2026-03-09 08:44:43,027 | INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-03-09 08:44:43,686 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 08:44:43,689 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-09 08:44:43,800 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-09 08:44:44,203 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 08:44:44,315 | INFO | HTTP Re

In [8]:
def get_repo_files(state):

    logger.info("Fetching repository file list")

    url = f"https://api.github.com/repos/{state['owner']}/{state['repo']}/git/trees/main?recursive=1"

    data = requests.get(url).json()

    files = [
        item["path"]
        for item in data["tree"]
        if item["type"] == "blob"
    ]

    logger.info(f"Total repo files: {len(files)}")

    return {"files": files}

In [9]:
def filter_relevant_files(state):

    logger.info("Filtering important files")

    extensions = (".py", ".js", ".ts", ".java")

    keywords = [
        "auth",
        "login",
        "security",
        "database",
        "api",
        "config"
    ]

    filtered = [
        f for f in state["files"]
        if f.endswith(extensions)
        and any(k in f.lower() for k in keywords)
    ]

    filtered = filtered[:10]  # hard limit

    logger.info(f"Files selected for analysis: {len(filtered)}")

    return {"filtered_files": filtered}

In [10]:
def fetch_files(state):

    logger.info("Downloading files")

    docs = []

    for file in state["filtered_files"]:

        url = f"https://raw.githubusercontent.com/{state['owner']}/{state['repo']}/main/{file}"

        try:
            text = requests.get(url).text

            docs.append({
                "file": file,
                "content": text
            })

        except Exception:
            logger.warning(f"Failed downloading {file}")

    logger.info(f"Files downloaded: {len(docs)}")

    return {"documents": docs}

In [11]:
def chunk_code(state):

    logger.info("Chunking code")

    chunks = []

    for doc in state["documents"]:

        lines = doc["content"].split("\n")

        for i in range(0, len(lines), 200):

            chunk = "\n".join(lines[i:i+200])

            chunks.append({
                "file": doc["file"],
                "content": chunk
            })

    logger.info(f"Chunks created: {len(chunks)}")

    return {"documents": chunks}

In [12]:
def build_vector_index(state):

    logger.info("Building vector index")

    texts = [d["content"] for d in state["documents"]]

    embeddings = embed_model.encode(texts)

    dim = len(embeddings[0])

    index = faiss.IndexFlatL2(dim)

    index.add(np.array(embeddings))

    logger.info("Vector DB ready")

    return {
        "embeddings": embeddings,
        "vector_index": index
    }

In [13]:
def retrieve_security_chunks(state):

    logger.info("Retrieving security-related chunks")

    queries = [
        "authentication vulnerability",
        "sql injection risk",
        "hardcoded password api key",
        "input validation vulnerability"
    ]

    results = []

    for q in queries:

        q_embedding = embed_model.encode([q])

        D, I = state["vector_index"].search(q_embedding, 2)

        for idx in I[0]:
            results.append(state["documents"][idx])

    logger.info(f"Chunks retrieved for scanning: {len(results)}")

    return {"retrieved_chunks": results}

In [14]:
def scan_code(state):

    logger.info("Running vulnerability scan")

    combined_code = "\n\n".join(
        chunk["content"] for chunk in state["retrieved_chunks"]
    )

    prompt = f"""
You are a security code auditor.

Find vulnerabilities in this code and explain briefly.

Code:
{combined_code}
"""

    output = llm(prompt)[0]["generated_text"]

    issues = []

    if "vulnerab" in output.lower():
        issues.append({
            "description": output
        })

    logger.info(f"Issues detected: {len(issues)}")

    return {"issues": issues}

In [15]:
def compute_repo_score(state):

    logger.info("Computing karma score")

    base_score = 100

    penalty = len(state["issues"]) * 10

    score = max(base_score - penalty, 0)

    logger.info(f"Karma Score: {score}")

    return {"karma_score": score}

In [16]:
graph = StateGraph(RepoState)

graph.add_node("get_repo_files", get_repo_files)
graph.add_node("filter_relevant_files", filter_relevant_files)
graph.add_node("fetch_files", fetch_files)
graph.add_node("chunk_code", chunk_code)
graph.add_node("build_vector_index", build_vector_index)
graph.add_node("retrieve_security_chunks", retrieve_security_chunks)
graph.add_node("scan_code", scan_code)
graph.add_node("compute_repo_score", compute_repo_score)

graph.set_entry_point("get_repo_files")

graph.add_edge("get_repo_files", "filter_relevant_files")
graph.add_edge("filter_relevant_files", "fetch_files")
graph.add_edge("fetch_files", "chunk_code")
graph.add_edge("chunk_code", "build_vector_index")
graph.add_edge("build_vector_index", "retrieve_security_chunks")
graph.add_edge("retrieve_security_chunks", "scan_code")
graph.add_edge("scan_code", "compute_repo_score")

In [17]:
app = graph.compile()
result = app.invoke({
    "owner": "Vedant122003",
    "repo": "Screenshot-sorter-tool",
    "issues": []
})

print("Karma Score:", result["karma_score"])

2026-03-09 08:46:43,121 | INFO | Fetching repository file list
2026-03-09 08:46:44,508 | INFO | Total repo files: 61
2026-03-09 08:46:44,511 | INFO | Filtering important files
2026-03-09 08:46:44,513 | INFO | Files selected for analysis: 8
2026-03-09 08:46:44,516 | INFO | Downloading files
2026-03-09 08:46:52,313 | INFO | Files downloaded: 8
2026-03-09 08:46:52,316 | INFO | Chunking code
2026-03-09 08:46:52,318 | INFO | Chunks created: 8
2026-03-09 08:46:52,320 | INFO | Building vector index
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]
2026-03-09 08:46:52,756 | INFO | Vector DB ready
2026-03-09 08:46:52,757 | INFO | Retrieving security-related chunks
Batches: 100%|██████████| 1/1 [00:00<00:00, 125.16it/s]
2026-03-09 08:46:52,850 | INFO | Chunks retrieved for scanning: 8
2026-03-09 08:46:52,851 | INFO | Running vulnerability scan
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation 

Karma Score: 90
